# Sprint 3 — Full MoE Pipeline (single-session, end-to-end)

Runs cache rebuild → Phase 1 → Phase 2 → Phase 3 → Eval all in one session.

## Datasets to attach
| Dataset slug | Required |
|---|---|
| `iahmedhabib/medsam-vit-b` | ✅ |
| `mabdullahi454/tb-pipeline-checkpoints` | ✅ |
| `iahmedhabib/montgomery` | ✅ |
| `iahmedhabib/shehzhenn` | ✅ |
| `usmanshams/tbx-11` | ✅ |
| `organizations/nih-chest-xrays` | optional (sanity gate) |

**GPU:** T4 × 1  
**Estimated total runtime:** ~3.5–4 hrs
- Setup: ~3 min
- Phase 0 cache build: ~45 min
- Phase 1 expert pretraining: ~50 min
- Phase 2 joint MoE: ~65 min
- Phase 3 boundary critic: ~15 min
- Evaluation: ~35 min
- Backup zip: ~3 min

## Section 1 — Setup

In [1]:
# ── Cell 1.1: Clone repo and install dependencies ────────────────────────────
import os, subprocess, sys
from pathlib import Path

REPO_URL  = "https://github.com/mabdullahi7780/dl-project-codebase.git"  # <-- UPDATE if your fork is elsewhere
REPO_DIR  = "/kaggle/working/repo"
CACHE_DIR = "/kaggle/working/moe_cache_v2"
SAVE_DIR  = "/kaggle/working/moe_v2"
EVAL_DIR  = "/kaggle/working/eval_moe_v2"

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
Path(EVAL_DIR).mkdir(parents=True, exist_ok=True)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     os.path.join(REPO_DIR, "requirements.txt")],
    check=True
)
print("Setup complete.")

Cloning into '/kaggle/working/repo'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.8/529.8 kB 31.0 MB/s eta 0:00:00
Setup complete.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.2.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires jupyter-client<8.0,>=5.2.0, but you have jupyter-client 8.8.0 which is incompatible.


In [2]:
# ── Cell 1.2: Verify mounts ──────────────────────────────────────────────────
from pathlib import Path

MEDSAM_CKPT = Path("/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth")
C1_ADAPTER  = Path("/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component1_adapters.safetensors")
C4_DECODER  = Path("/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component4_mask_decoder.pt")

MONTGOMERY  = Path("/kaggle/input/datasets/iahmedhabib/montgomery/montgomery")
SHENZHEN    = Path("/kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen")
TBX11K      = Path("/kaggle/input/datasets/usmanshams/tbx-11/TBX11K")
NIH         = Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data")

def check(label, path, required=True):
    status = "OK" if path.exists() else "MISSING"
    flag   = "[REQUIRED]" if required else "[optional]"
    print(f"  {flag} {label:<40}: {status}  ({path})")
    if required and not path.exists():
        raise FileNotFoundError(f"{label} not found at {path}")

check("MedSAM checkpoint",  MEDSAM_CKPT)
check("C1 LoRA adapter",    C1_ADAPTER)
check("C4 lung decoder",    C4_DECODER)
check("Montgomery dataset", MONTGOMERY)
check("Shenzhen dataset",   SHENZHEN)
check("TBX11K dataset",     TBX11K)
check("NIH-CXR14 dataset",  NIH, required=False)
print("All required mounts present.")

  [REQUIRED] MedSAM checkpoint                       : OK  (/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth)
  [REQUIRED] C1 LoRA adapter                         : OK  (/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component1_adapters.safetensors)
  [REQUIRED] C4 lung decoder                         : OK  (/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component4_mask_decoder.pt)
  [REQUIRED] Montgomery dataset                      : OK  (/kaggle/input/datasets/iahmedhabib/montgomery/montgomery)
  [REQUIRED] Shenzhen dataset                        : OK  (/kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen)
  [REQUIRED] TBX11K dataset                          : OK  (/kaggle/input/datasets/usmanshams/tbx-11/TBX11K)
  [optional] NIH-CXR14 dataset                       : OK  (/kaggle/input/datasets/organizations/nih-chest-xrays/data)
All required mounts present.


In [3]:
# ── Cell 1.3: Patch configs ──────────────────────────────────────────────────
import yaml

moe_yaml_path   = Path(REPO_DIR) / "configs" / "moe.yaml"
paths_yaml_path = Path(REPO_DIR) / "configs" / "paths.yaml"

with moe_yaml_path.open() as f:
    moe_cfg = yaml.safe_load(f)

moe_cfg["component1"]["checkpoint_path"]         = str(MEDSAM_CKPT)
moe_cfg["component1"]["adapter_path"]             = str(C1_ADAPTER)
moe_cfg["component4"]["checkpoint_path"]         = str(MEDSAM_CKPT)
moe_cfg["component4"]["decoder_checkpoint_path"] = str(C4_DECODER)
moe_cfg["moe_training"]["save_dir"]              = SAVE_DIR
# Guard: ensure v2 settings are active even if YAML drifts
moe_cfg["moe_training"]["joint"]["gate_only"]            = False
moe_cfg["moe_training"]["joint"]["epochs"]               = 12
moe_cfg["moe_training"]["joint"]["lr_gate"]              = 5e-4
moe_cfg["moe_training"]["joint"]["lr_experts"]           = 5e-5
moe_cfg["moe_training"]["joint"]["load_balance_weight"]  = 0.05
moe_cfg["moe_training"]["joint"]["expert_aux_weight"]    = 0.5
moe_cfg["moe_training"]["joint"]["resume_from_moe_best"] = None
moe_cfg["moe"]["checkpoint_path"]                         = str(Path(SAVE_DIR) / "moe_best.pt")

with moe_yaml_path.open("w") as f:
    yaml.safe_dump(moe_cfg, f, sort_keys=False)

paths_cfg = {
    "project_root": REPO_DIR,
    "datasets": {
        "montgomery": str(MONTGOMERY),
        "shenzhen":   str(SHENZHEN),
        "tbx11k":     str(TBX11K),
        "nih_cxr14":  str(NIH) if NIH.exists() else "",
    },
}
with paths_yaml_path.open("w") as f:
    yaml.safe_dump(paths_cfg, f, sort_keys=False)

print("Configs patched.")
print(f"  Phase 2: gate_only={moe_cfg['moe_training']['joint']['gate_only']}, epochs={moe_cfg['moe_training']['joint']['epochs']}")

Configs patched.
  Phase 2: gate_only=False, epochs=12


## Section 2 — Phase 0: Cache Rebuild (~45 min)

Run smoke test first (~3 min). If it shows NIH ALP < 10%, proceed to full build.

In [4]:
# ── Cell 2.1: Smoke cache build (8 images per domain, ~3 min) ───────────────
import subprocess, sys
from pathlib import Path

SMOKE_CACHE = "/kaggle/working/moe_cache_smoke"

cmd = [
    sys.executable, "-m", "scripts.cache_moe_embeddings",
    "--config",          str(Path(REPO_DIR) / "configs" / "moe.yaml"),
    "--paths",           str(Path(REPO_DIR) / "configs" / "paths.yaml"),
    "--output",          SMOKE_CACHE,
    "--limit-per-domain","8",
]
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=False)
assert result.returncode == 0, "Smoke cache build failed — check output above."
smoke_files = list(Path(SMOKE_CACHE).glob("*.pt"))
print(f"Smoke cache: {len(smoke_files)} files written to {SMOKE_CACHE}")

If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
[cache] datasets=['tbx11k', 'shenzhen', 'montgomery'] samples=24 output=/kaggle/working/moe_cache_smoke

Cache complete.
  output_dir: /kaggle/working/moe_cache_smoke
  per_dataset: {'montgomery': 8, 'shenzhen': 8, 'tbx11k': 8}
  supervision: {'pseudo_cam': 16, 'tbx_bbox': 8}
Smoke cache: 24 files written to /kaggle/working/moe_cache_smoke


In [5]:
# ── Cell 2.2: Smoke sanity check ────────────────────────────────────────────
# EXPECTED non-zero target fractions:
#   consolidation : 25–45%
#   cavity        :  8–20%  (was 0–2% with the broken ringify + whole-lung fallback)
#   fibrosis      : 15–35%
#   nodule        : 15–30%
# NIH is intentionally excluded from the cache (per CLAUDE.md). The real
# NIH "lung-painter" gate runs at eval time (Cell 6.5).

import torch
from pathlib import Path

expert_names = ["consolidation", "cavity", "fibrosis", "nodule"]
stats = {e: {"total": 0, "nonzero": 0} for e in expert_names}

for p in Path(SMOKE_CACHE).glob("*.pt"):
    d = torch.load(p, weights_only=False)
    for e in expert_names:
        mask = d["expert_masks"][e]
        has_content = float(mask.sum().item()) > 0
        stats[e]["total"]   += 1
        stats[e]["nonzero"] += int(has_content)

print("Expert non-zero mask fractions in smoke cache:")
GATE_PASS = True
for e in expert_names:
    t = stats[e]["total"]
    n = stats[e]["nonzero"]
    pct = 100.0 * n / t if t else 0.0
    ok = pct > 0  # Some non-zero on at least the smoke set; full check is in Cell 2.4
    if not ok: GATE_PASS = False
    print(f"  [{'OK  ' if ok else 'WARN'}] {e:<16}: {n}/{t}  ({pct:.1f}%)")

assert GATE_PASS, "Smoke cache: every expert is empty — supervision pipeline broken."
print("\nSmoke OK. Proceeding to full build.")


Expert non-zero mask fractions in smoke cache:
  [OK  ] consolidation   : 24/24  (100.0%)
  [OK  ] cavity          : 22/24  (91.7%)
  [OK  ] fibrosis        : 24/24  (100.0%)
  [OK  ] nodule          : 24/24  (100.0%)

Smoke OK. Proceeding to full build.


In [6]:
# ── Cell 2.3: Full cache build (1000 images per domain, ~45 min) ────────────
import subprocess, sys, time
from pathlib import Path

t0 = time.time()
cmd = [
    sys.executable, "-m", "scripts.cache_moe_embeddings",
    "--config",          str(Path(REPO_DIR) / "configs" / "moe.yaml"),
    "--paths",           str(Path(REPO_DIR) / "configs" / "paths.yaml"),
    "--output",          CACHE_DIR,
    "--limit-per-domain","1000",
]
result = subprocess.run(cmd, cwd=REPO_DIR)
elapsed = (time.time() - t0) / 60
assert result.returncode == 0, "Full cache build failed — check output above."

cache_files = list(Path(CACHE_DIR).glob("*.pt"))
print(f"Full cache: {len(cache_files)} files written in {elapsed:.1f} min")

[cache] datasets=['tbx11k', 'shenzhen', 'montgomery'] samples=1800 output=/kaggle/working/moe_cache_v2
  cached 50/1800
  cached 100/1800
  cached 150/1800
  cached 200/1800
  cached 250/1800
  cached 300/1800
  cached 350/1800
  cached 400/1800
  cached 450/1800
  cached 500/1800
  cached 550/1800
  cached 600/1800
  cached 650/1800
  cached 700/1800
  cached 750/1800
  cached 800/1800
  cached 850/1800
  cached 900/1800
  cached 950/1800
  cached 1000/1800
  cached 1050/1800
  cached 1100/1800
  cached 1150/1800
  cached 1200/1800
  cached 1250/1800
  cached 1300/1800
  cached 1350/1800
  cached 1400/1800
  cached 1450/1800
  cached 1500/1800
  cached 1550/1800
  cached 1600/1800
  cached 1650/1800
  cached 1700/1800
  cached 1750/1800
  cached 1800/1800

Cache complete.
  output_dir: /kaggle/working/moe_cache_v2
  per_dataset: {'montgomery': 138, 'shenzhen': 662, 'tbx11k': 1000}
  supervision: {'pseudo_cam': 800, 'tbx_bbox': 799, 'tbx_pseudo_cam': 201}
Full cache: 1800 files written

In [7]:
# ── Cell 2.4: Full cache sanity check + dataset balance ─────────────────────
import torch
from collections import Counter
from pathlib import Path

expert_names = ["consolidation", "cavity", "fibrosis", "nodule"]
dataset_counts = Counter()
expert_nonzero = {e: Counter() for e in expert_names}

for p in Path(CACHE_DIR).glob("*.pt"):
    ds = p.stem.split("__")[0]
    dataset_counts[ds] += 1
    d = torch.load(p, weights_only=False)
    for e in expert_names:
        if float(d["expert_masks"][e].sum().item()) > 0:
            expert_nonzero[e][ds] += 1

print("Dataset sample counts (balanced sampler will equalise during training):")
for ds, cnt in sorted(dataset_counts.items()):
    print(f"  {ds:<20}: {cnt}")

total = sum(dataset_counts.values())
print("\nExpert non-zero target fractions (all datasets):")
GATE_PASS = True
for e in expert_names:
    n = sum(expert_nonzero[e].values())
    pct = 100.0 * n / total if total else 0.0
    ok = pct > 2.0
    if not ok: GATE_PASS = False
    print(f"  [{'OK  ' if ok else 'FAIL'}] {e:<16}: {n}/{total}  ({pct:.1f}%)")

assert GATE_PASS, "Cache sanity check FAILED — do not proceed to training."
print("\nFull cache passed sanity check.")


Dataset sample counts (balanced sampler will equalise during training):
  montgomery          : 138
  shenzhen            : 662
  tbx11k              : 1000

Expert non-zero target fractions (all datasets):
  [OK  ] consolidation   : 1800/1800  (100.0%)
  [OK  ] cavity          : 1481/1800  (82.3%)
  [OK  ] fibrosis        : 1800/1800  (100.0%)
  [OK  ] nodule          : 1800/1800  (100.0%)

Full cache passed sanity check.


## Section 3 — Phase 1: Expert Pretraining (~50 min)

In [8]:
# ── Cell 3.1: Phase 1 — pretrain all 4 expert decoders ──────────────────────
import subprocess, sys, time
from pathlib import Path

t0 = time.time()
result = subprocess.run(
    [
        sys.executable, "-m", "src.training.train_experts",
        "--config",    str(Path(REPO_DIR) / "configs" / "moe.yaml"),
        "--cache-dir", CACHE_DIR,
        "--all",
    ],
    cwd=REPO_DIR,
)
elapsed = (time.time() - t0) / 60
print(f"Phase 1 finished in {elapsed:.1f} min  (return code: {result.returncode})")
assert result.returncode == 0, "Phase 1 failed — check output above."

# train_experts.py saves <expert>_best.pt AND <expert>_final.pt per expert (8 files total)
best_ckpts = sorted(Path(SAVE_DIR).glob("expert_*_best.pt"))
print(f"Best expert checkpoints saved ({len(best_ckpts)}):")
for p in best_ckpts:
    print(f"  {p.name}: {p.stat().st_size/1e6:.1f} MB")
assert len(best_ckpts) == 4, f"Expected 4 *_best.pt files, found {len(best_ckpts)}"


[Expert pretrain] consolidation: 10 epochs, bs=4, lr=0.0001
  epoch 0: loss=0.7373
  epoch 1: loss=0.6444
  epoch 2: loss=0.5751
  epoch 3: loss=0.5246
  epoch 4: loss=0.4902
  epoch 5: loss=0.4667
  epoch 6: loss=0.4490
  epoch 7: loss=0.4404
  epoch 8: loss=0.4311
  epoch 9: loss=0.4313
  Expert consolidation saved to /kaggle/working/moe_v2/expert_consolidation_final.pt
[Expert pretrain] cavity: 10 epochs, bs=4, lr=0.0001
  epoch 0: loss=0.6942
  epoch 1: loss=0.6202
  epoch 2: loss=0.5676
  epoch 3: loss=0.5295
  epoch 4: loss=0.4982
  epoch 5: loss=0.4774
  epoch 6: loss=0.4676
  epoch 7: loss=0.4564
  epoch 8: loss=0.4524
  epoch 9: loss=0.4476
  Expert cavity saved to /kaggle/working/moe_v2/expert_cavity_final.pt
[Expert pretrain] fibrosis: 10 epochs, bs=4, lr=0.0001
  epoch 0: loss=0.6919
  epoch 1: loss=0.6071
  epoch 2: loss=0.5479
  epoch 3: loss=0.5110
  epoch 4: loss=0.4805
  epoch 5: loss=0.4653
  epoch 6: loss=0.4499
  epoch 7: loss=0.4410
  epoch 8: loss=0.4323
  epoch 9

## Section 4 — Phase 2: Joint MoE Training (~65 min)

In [9]:
# ── Cell 4.1: Phase 2 — joint training (gate + experts + fusion) ────────────
import subprocess, sys, time
from pathlib import Path

t0 = time.time()
result = subprocess.run(
    [
        sys.executable, "-m", "src.training.train_moe_joint",
        "--config",    str(Path(REPO_DIR) / "configs" / "moe.yaml"),
        "--cache-dir", CACHE_DIR,
    ],
    cwd=REPO_DIR,
)
elapsed = (time.time() - t0) / 60
print(f"Phase 2 finished in {elapsed:.1f} min  (return code: {result.returncode})")
assert result.returncode == 0, "Phase 2 failed — check output above."

moe_best = Path(SAVE_DIR) / "moe_best.pt"
assert moe_best.exists(), "moe_best.pt not found after Phase 2!"
print(f"moe_best.pt saved: {moe_best.stat().st_size / 1e6:.1f} MB")

  Loaded pretrained expert: consolidation from /kaggle/working/moe_v2/expert_consolidation_best.pt
  Loaded pretrained expert: cavity from /kaggle/working/moe_v2/expert_cavity_best.pt
  Loaded pretrained expert: fibrosis from /kaggle/working/moe_v2/expert_fibrosis_best.pt
  Loaded pretrained expert: nodule from /kaggle/working/moe_v2/expert_nodule_best.pt
[Joint MoE] 12 epochs, bs=4
  LR: gate=0.0005, experts=5e-05, fusion=5e-05, expert_aux=0.5, use_domain_ctx=True
  epoch 0: total=0.6879  mask=0.4116  expert=0.4511  lb=1.0160
  epoch 1: total=0.6712  mask=0.3993  expert=0.4429  lb=1.0092
  epoch 2: total=0.6570  mask=0.3882  expert=0.4368  lb=1.0067
  epoch 3: total=0.6558  mask=0.3877  expert=0.4354  lb=1.0073
  epoch 4: total=0.6545  mask=0.3862  expert=0.4353  lb=1.0130
  epoch 5: total=0.6467  mask=0.3798  expert=0.4319  lb=1.0188
  epoch 6: total=0.6346  mask=0.3704  expert=0.4261  lb=1.0240
  epoch 7: total=0.6324  mask=0.3674  expert=0.4257  lb=1.0446
  epoch 8: total=0.6353  m

In [10]:
# ── Cell 4.2: Phase 2 convergence check ─────────────────────────────────────
# Healthy: mask_loss drops by > 0.03 across 12 epochs.
# Healthy: lb_loss drops toward 1.0 (gate is specialising, not routing uniformly).
import json
from pathlib import Path

log_file = Path(SAVE_DIR) / "moe_joint_history.jsonl"
if log_file.exists():
    epochs, mask_losses, lb_losses, total_losses = [], [], [], []
    with log_file.open() as f:
        for line in f:
            e = json.loads(line)
            epochs.append(e.get("epoch"))
            mask_losses.append(e.get("mask_loss", float("nan")))
            lb_losses.append(e.get("lb_loss", float("nan")))
            total_losses.append(e.get("total", float("nan")))

    print(f"  {'Epoch':<8} {'total':<10} {'mask_loss':<12} {'lb_loss':<10}")
    for ep, tot, ml, lb in zip(epochs, total_losses, mask_losses, lb_losses):
        print(f"  {ep:<8} {tot:<10.4f} {ml:<12.4f} {lb:<10.4f}")

    if len(mask_losses) >= 2:
        delta = mask_losses[0] - mask_losses[-1]
        print(f"\nMask loss drop: {delta:.4f}  ({'OK — model learning' if delta > 0.03 else 'WARN — barely moved'})")
        lb_drop = lb_losses[0] - lb_losses[-1]
        print(f"LB loss drop:   {lb_drop:.4f}  ({'OK — gate specialising' if lb_drop > 0.05 else 'WARN — gate uniform'})")
else:
    print(f"Log file not found at {log_file} — training may still have succeeded.")


  Epoch    total      mask_loss    lb_loss   
  0        0.6879     0.4116       1.0160    
  1        0.6712     0.3993       1.0092    
  2        0.6570     0.3882       1.0067    
  3        0.6558     0.3877       1.0073    
  4        0.6545     0.3862       1.0130    
  5        0.6467     0.3798       1.0188    
  6        0.6346     0.3704       1.0240    
  7        0.6324     0.3674       1.0446    
  8        0.6353     0.3686       1.0457    
  9        0.6264     0.3616       1.0514    
  10       0.6203     0.3572       1.0499    
  11       0.6242     0.3596       1.0529    

Mask loss drop: 0.0519  (OK — model learning)
LB loss drop:   -0.0368  (WARN — gate uniform)


## Section 5 — Phase 3: Boundary Critic (~15 min, optional)

In [11]:
# ── Cell 5.1: Phase 3 — boundary critic ─────────────────────────────────────
import subprocess, sys, time
from pathlib import Path

t0 = time.time()
result = subprocess.run(
    [
        sys.executable, "-m", "src.training.train_boundary_critic",
        "--config",    str(Path(REPO_DIR) / "configs" / "moe.yaml"),
        "--cache-dir", CACHE_DIR,
    ],
    cwd=REPO_DIR,
)
elapsed = (time.time() - t0) / 60
print(f"Phase 3 finished in {elapsed:.1f} min  (return code: {result.returncode})")
if result.returncode != 0:
    print("Phase 3 failed — boundary critic is optional; proceeding to eval.")

[Boundary critic] 5 epochs, bs=16, lr=0.001
  epoch 0: loss=0.6926 acc=0.5606
  epoch 1: loss=0.6316 acc=0.6328
  epoch 2: loss=0.6112 acc=0.6417
  epoch 3: loss=0.5946 acc=0.6542
  epoch 4: loss=0.5782 acc=0.6769
Boundary critic saved to /kaggle/working/moe_v2/boundary_critic.pt
Phase 3 finished in 1.7 min  (return code: 0)


## Section 6 — Evaluation (~35 min)

In [12]:
# ── Cell 6.1: Build eval-specific configs ───────────────────────────────────
import yaml
from pathlib import Path

moe_eval_cfg = yaml.safe_load(open(Path(REPO_DIR) / "configs" / "moe.yaml"))
moe_eval_cfg["moe"]["checkpoint_path"] = str(Path(SAVE_DIR) / "moe_best.pt")

moe_eval_file = Path(EVAL_DIR) / "moe_eval.yaml"
with moe_eval_file.open("w") as f:
    yaml.safe_dump(moe_eval_cfg, f, sort_keys=False)

paths_eval_cfg = {
    "project_root": REPO_DIR,
    "datasets": {
        "montgomery": str(MONTGOMERY),
        "shenzhen":   str(SHENZHEN),
        "tbx11k":     str(TBX11K),
        "nih_cxr14":  str(NIH) if NIH.exists() else "",
    },
}
paths_eval_file = Path(EVAL_DIR) / "paths_eval.yaml"
with paths_eval_file.open("w") as f:
    yaml.safe_dump(paths_eval_cfg, f, sort_keys=False)

print("Eval configs written.")

Eval configs written.


In [13]:
# ── Cell 6.2: Run MoE evaluation ────────────────────────────────────────────
# tbx_list_name="all_trainval.txt" is the safe default that exists in the dataset.
# (test.txt may not be present in this Kaggle mount; falls back to dir walk.)
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.evaluation.moe_eval import run_moe_evaluation

summary = run_moe_evaluation(
    moe_config_path  = Path(EVAL_DIR) / "moe_eval.yaml",
    paths_config_path= Path(EVAL_DIR) / "paths_eval.yaml",
    output_dir       = Path(EVAL_DIR),
    limit_per_domain = 200,
    tbx_list_name    = "all_trainval.txt",
    repo_root        = Path(REPO_DIR),
)
print("\nMoE evaluation complete.")


MoE eval device: cuda:Tesla T4
Building manifests …
  montgomery : 138 total
  shenzhen   : 662 total
  tbx11k     : 8976 total
  nih_cxr14  : 112120 total
  held-out montgomery : 28 images
  held-out shenzhen   : 132 images
  held-out tbx11k     : 200 images
  held-out nih_cxr14  : 200 images
Loading ground truth …
  NIH multilabels: 112120
  TBX11K bbox GT : 799
Building models …
Component 1: loaded LoRA+DANN adapters from /kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component1_adapters.safetensors
Component 4: loaded fine-tuned decoder from /kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component4_mask_decoder.pt
MoE: loaded checkpoint from /kaggle/working/moe_v2/moe_best.pt
Running MoE pipeline on held-out images …
  [  20/560] elapsed=  27.5s rate=0.73 img/s eta= 741.4s
  [  40/560] elapsed=  51.8s rate=0.77 img/s eta= 674.0s
  [  60/560] elapsed=  74.4s rate=0.81 img/s eta= 619.7s
  [  80/560] elapsed=  97.6s rate=0.82 img/s eta= 585.5s
  [ 100/560] 

In [14]:
# ── Cell 6.3: Component metrics ─────────────────────────────────────────────
import pandas as pd
from pathlib import Path

comp_df = pd.read_csv(Path(EVAL_DIR) / "moe_components.csv")
comp_df["value"] = comp_df["value"].apply(
    lambda x: f"{float(x):.4f}" if pd.notna(x) and str(x) != "nan" else "N/A"
)
print("=== MoE COMPONENT-LEVEL METRICS ===")
print(comp_df[["metric","dataset","value","n","notes"]].to_string(index=False))

=== MoE COMPONENT-LEVEL METRICS ===
                  metric    dataset  value   n                           notes
      c1_domain_accuracy montgomery 0.0000  28                             NaN
      c1_domain_accuracy   shenzhen 0.0000 132                             NaN
      c1_domain_accuracy     tbx11k 0.0000 200                             NaN
      c1_domain_accuracy  nih_cxr14 1.0000 200                             NaN
      c1_domain_accuracy    overall 0.3571 560 target=0.25 (chance) after DANN
            c4_lung_dice montgomery 0.8862  28                             NaN
             c4_lung_iou montgomery 0.8033  28                             NaN
            c4_lung_dice   shenzhen 0.9592 110                             NaN
             c4_lung_iou   shenzhen 0.9222 110                             NaN
c2_pathology_macro_auroc  nih_cxr14 0.6848 200   macro over 8 TB-mimic classes
         moe_lesion_dice     tbx11k    N/A   0       no TBX11K bbox GT matched


In [15]:
# ── Cell 6.4: System metrics ────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

sys_df = pd.read_csv(Path(EVAL_DIR) / "moe_system.csv")
sys_df["value"] = sys_df["value"].apply(
    lambda x: f"{float(x):.4f}" if pd.notna(x) and str(x) != "nan" else "N/A"
)
print("=== MoE SYSTEM-LEVEL METRICS ===")
print(sys_df[["metric","dataset","value","n","notes"]].to_string(index=False))

=== MoE SYSTEM-LEVEL METRICS ===
               metric    dataset   value   n                                                 notes
             alp_mean montgomery 29.6092  28                                                   NaN
              alp_std montgomery  9.2584  28                                                   NaN
              alp_p50 montgomery 29.6141  28                                                   NaN
              alp_p95 montgomery 46.6306  28                                                   NaN
             alp_mean   shenzhen 53.8827 132                                                   NaN
              alp_std   shenzhen  9.7898 132                                                   NaN
              alp_p50   shenzhen 54.8504 132                                                   NaN
              alp_p95   shenzhen 66.6164 132                                                   NaN
             alp_mean     tbx11k 39.9083 200                                

In [16]:
# ── Cell 6.5: NIH ALP sanity gate ───────────────────────────────────────────
# If NIH ALP mean > 8%, experts are still painting lungs. The fix didn't take.
import pandas as pd
from pathlib import Path

pi_df  = pd.read_csv(Path(EVAL_DIR) / "moe_per_image.csv")
nih_df = pi_df[pi_df["dataset_id"] == "nih_cxr14"]

if nih_df.empty:
    print("NIH-CXR14 not in eval (dataset not attached). Sanity gate skipped.")
else:
    alp_mean = nih_df["alp"].mean()
    alp_p95  = nih_df["alp"].quantile(0.95)
    ok = alp_mean < 8.0
    print(f"NIH-CXR14 ALP (n={len(nih_df)}):")
    print(f"  mean : {alp_mean:.2f}%  ({'OK' if ok else 'FAIL — experts painting lungs'})")
    print(f"  p95  : {alp_p95:.2f}%")

NIH-CXR14 ALP (n=200):
  mean : 30.72%  (FAIL — experts painting lungs)
  p95  : 54.01%


In [17]:
# ── Cell 6.6: Bootstrap 95% CI on Timika AUROC ──────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score

pi_df = pd.read_csv(Path(EVAL_DIR) / "moe_per_image.csv")

def bootstrap_auroc(y_true, y_score, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    auc_vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt, ys = y_true[idx], y_score[idx]
        if len(np.unique(yt)) < 2:
            continue
        try:
            auc_vals.append(roc_auc_score(yt, ys))
        except Exception:
            pass
    if not auc_vals:
        return float("nan"), float("nan"), float("nan")
    return float(np.mean(auc_vals)), float(np.percentile(auc_vals, 2.5)), float(np.percentile(auc_vals, 97.5))

TARGET = {"montgomery": 0.60, "shenzhen": 0.70, "tbx11k": 0.70}

print("Timika AUROC with 95% bootstrap CI (2000 resamples):")
print(f"  {'Dataset':<14} {'AUROC':>8}  {'95% CI':>16}  {'n_total':>8}  {'n_TB+':>7}  {'Status'}")
print("  " + "-" * 75)
for dom in ["montgomery", "shenzhen", "tbx11k"]:
    sub = pi_df[(pi_df["dataset_id"] == dom) & pi_df["tb_label"].notna()].copy()
    if sub.empty or sub["tb_label"].nunique() < 2:
        print(f"  {dom:<14}   N/A      (no/single-class labels, n={len(sub)})")
        continue
    y_true  = sub["tb_label"].astype(int).values
    y_score = sub["timika_score"].values
    mean_auc, lo, hi = bootstrap_auroc(y_true, y_score)
    n_pos = int(y_true.sum())
    target = TARGET.get(dom, 0.70)
    met = "PASS" if mean_auc >= target else "FAIL"
    print(f"  {dom:<14}  {mean_auc:>6.4f}   [{lo:.4f}–{hi:.4f}]  {len(sub):>8}  {n_pos:>7}  [{met} (≥{target})]")

Timika AUROC with 95% bootstrap CI (2000 resamples):
  Dataset           AUROC            95% CI   n_total    n_TB+  Status
  ---------------------------------------------------------------------------
  montgomery      0.4038   [0.1871–0.6257]        28       14  [FAIL (≥0.6)]
  shenzhen        0.6230   [0.5243–0.7199]       132       62  [FAIL (≥0.7)]
  tbx11k           N/A      (no/single-class labels, n=82)


In [18]:
# ── Cell 6.7: Sprint 3 target table ─────────────────────────────────────────
import pandas as pd
from pathlib import Path

sys_df = pd.read_csv(Path(EVAL_DIR) / "moe_system.csv")
sys_df["value"] = pd.to_numeric(sys_df["value"], errors="coerce")

TARGETS = {
    ("timika_auroc",  "montgomery"): ("≥ 0.60", 0.60),
    ("timika_auroc",  "shenzhen"):   ("≥ 0.70", 0.70),
    ("tb_head_auroc", "shenzhen"):   ("≥ 0.85", 0.85),
    ("tb_head_auroc", "montgomery"): ("≥ 0.80", 0.80),
}

print(f"{'Metric':<25} {'Dataset':<14} {'Achieved':>10}  {'Target':>8}  {'Status'}")
print("-" * 70)
for (metric, dataset), (target_str, target_val) in TARGETS.items():
    row = sys_df[(sys_df["metric"] == metric) & (sys_df["dataset"] == dataset)]
    if row.empty or pd.isna(row["value"].values[0]):
        print(f"{metric:<25} {dataset:<14}      N/A      {target_str:>8}  N/A")
    else:
        val = float(row["value"].values[0])
        status = "PASS" if val >= target_val else f"FAIL (gap={target_val-val:.4f})"
        print(f"{metric:<25} {dataset:<14}  {val:>8.4f}    {target_str:>8}  {status}")

Metric                    Dataset          Achieved    Target  Status
----------------------------------------------------------------------
timika_auroc              montgomery        0.4031      ≥ 0.60  FAIL (gap=0.1969)
timika_auroc              shenzhen          0.6217      ≥ 0.70  FAIL (gap=0.0783)
tb_head_auroc             shenzhen          0.8802      ≥ 0.85  PASS
tb_head_auroc             montgomery        0.4337      ≥ 0.80  FAIL (gap=0.3663)


## Section 7 — Backup Artifacts (zip cache + MoE weights)

In [19]:
# ── Cell 7.1: Zip MoE weights and eval results for download ────────────────
# NOTE: The cache itself (CACHE_DIR) is many GB. Set ZIP_CACHE=True only if
# you actually need to download it. The MoE weights and eval results are
# small enough to zip and download safely.
import shutil
from pathlib import Path
from IPython.display import FileLink, display

ZIP_CACHE = False  # Set True only if you really need to download the cache

print("Zipping MoE weights ...")
moe_zip = shutil.make_archive("/kaggle/working/moe_v2", "zip", SAVE_DIR)
print(f"  {moe_zip}  ({Path(moe_zip).stat().st_size/1e6:.1f} MB)")

print("Zipping eval results ...")
eval_zip = shutil.make_archive("/kaggle/working/eval_moe_v2", "zip", EVAL_DIR)
print(f"  {eval_zip}  ({Path(eval_zip).stat().st_size/1e6:.1f} MB)")

print("\nDownload links:")
display(FileLink(moe_zip))
display(FileLink(eval_zip))

if ZIP_CACHE:
    print("\nZipping cache (large, may take 5+ min) ...")
    cache_zip = shutil.make_archive("/kaggle/working/moe_cache_v2", "zip", CACHE_DIR)
    print(f"  {cache_zip}  ({Path(cache_zip).stat().st_size/1e6:.1f} MB)")
    display(FileLink(cache_zip))

print("\nAll done. Outputs in /kaggle/working/.")
print("To save as Kaggle dataset: click the 3-dot menu next to each .zip in the Output panel.")


Zipping MoE weights ...
  /kaggle/working/moe_v2.zip  (68.0 MB)
Zipping eval results ...
  /kaggle/working/eval_moe_v2.zip  (0.8 MB)

Download links:


/kaggle/working/moe_v2.zip

/kaggle/working/eval_moe_v2.zip


All done. Outputs in /kaggle/working/.
To save as Kaggle dataset: click the 3-dot menu next to each .zip in the Output panel.
